# MNIST Digit Classifier + Generator
A neural network that:
1. **Classifies** handwritten digits (0-9) with accuracy metrics
2. **Generates** new digit images using a GAN (Generative Adversarial Network)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 1. Load MNIST Dataset

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=2)

print(f"Train: {len(train_dataset)} images")
print(f"Test: {len(test_dataset)} images")

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    img, label = train_dataset[i]
    ax.imshow(img.squeeze(), cmap="gray")
    ax.set_title(f"Label: {label}", fontsize=12)
    ax.axis("off")
plt.suptitle("Sample MNIST Digits", fontsize=14)
plt.tight_layout()
plt.show()

## 2. Digit Classifier (CNN)

In [ ]:
class DigitClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

classifier = DigitClassifier().to(device)
params = sum(p.numel() for p in classifier.parameters())
print(f"Classifier params: {params:,}")
print(classifier)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(classifier.parameters(), lr=0.001)

EPOCHS = 5
for epoch in range(EPOCHS):
    classifier.train()
    total_loss = 0
    correct = 0
    total = 0
    for batch_idx, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = classifier(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total += labels.size(0)
    train_acc = 100. * correct / total
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {avg_loss:.4f} | Train Acc: {train_acc:.2f}%")

In [ ]:
classifier.eval()
correct = 0
total = 0
class_correct = [0] * 10
class_total = [0] * 10

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = classifier(images)
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total += labels.size(0)
        for i in range(labels.size(0)):
            label = labels[i].item()
            class_correct[label] += (predicted[i] == label).item()
            class_total[label] += 1

accuracy = 100. * correct / total
print(f"\nTest Accuracy: {accuracy:.2f}%")
print(f"\nPer-class accuracy:")
for i in range(10):
    acc = 100. * class_correct[i] / class_total[i]
    print(f"  Digit {i}: {acc:.1f}% ({class_correct[i]}/{class_total[i]})")

In [ ]:
classifier.eval()
images, labels = next(iter(test_loader))
images, labels = images.to(device), labels.to(device)
with torch.no_grad():
    outputs = classifier(images)
    _, preds = outputs.max(1)

fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for i, ax in enumerate(axes.flat):
    img = images[i].cpu().squeeze()
    true = labels[i].item()
    pred = preds[i].item()
    color = "green" if true == pred else "red"
    ax.imshow(img, cmap="gray")
    ax.set_title(f"True: {true} | Pred: {pred}", color=color, fontsize=11)
    ax.axis("off")
plt.suptitle("Classifier Predictions (Green=Correct, Red=Wrong)", fontsize=13)
plt.tight_layout()
plt.show()

## 3. Digit Generator (GAN)
Generates new handwritten digit images from random noise.

In [ ]:
class Generator(nn.Module):
    def __init__(self, latent_dim=100):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, 256),
            nn.LeakyReLU(0.2),
            nn.BatchNorm1d(256),
            nn.Linear(256, 512),
            nn.LeakyReLU(0.2),
            nn.BatchNorm1d(512),
            nn.Linear(512, 1024),
            nn.LeakyReLU(0.2),
            nn.BatchNorm1d(1024),
            nn.Linear(1024, 28 * 28),
            nn.Tanh()
        )

    def forward(self, z):
        return self.net(z).view(-1, 1, 28, 28)


class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 512),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(256, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)


LATENT_DIM = 100
generator = Generator(LATENT_DIM).to(device)
discriminator = Discriminator().to(device)

g_params = sum(p.numel() for p in generator.parameters())
d_params = sum(p.numel() for p in discriminator.parameters())
print(f"Generator params: {g_params:,}")
print(f"Discriminator params: {d_params:,}")

In [ ]:
criterion = nn.BCELoss()
opt_g = optim.Adam(generator.parameters(), lr=0.0002, betas=(0.5, 0.999))
opt_d = optim.Adam(discriminator.parameters(), lr=0.0002, betas=(0.5, 0.999))

EPOCHS_GAN = 30
for epoch in range(EPOCHS_GAN):
    d_loss_sum = 0
    g_loss_sum = 0
    for real_images, _ in train_loader:
        batch_size = real_images.size(0)
        real_images = real_images.to(device)
        real_labels = torch.ones(batch_size, 1).to(device)
        fake_labels = torch.zeros(batch_size, 1).to(device)

        z = torch.randn(batch_size, LATENT_DIM).to(device)
        fake_images = generator(z)

        d_real = discriminator(real_images)
        d_fake = discriminator(fake_images.detach())
        d_loss = criterion(d_real, real_labels) + criterion(d_fake, fake_labels)
        opt_d.zero_grad()
        d_loss.backward()
        opt_d.step()

        z = torch.randn(batch_size, LATENT_DIM).to(device)
        fake_images = generator(z)
        d_fake = discriminator(fake_images)
        g_loss = criterion(d_fake, real_labels)
        opt_g.zero_grad()
        g_loss.backward()
        opt_g.step()

        d_loss_sum += d_loss.item()
        g_loss_sum += g_loss.item()

    n = len(train_loader)
    print(f"Epoch {epoch+1}/{EPOCHS_GAN} | D Loss: {d_loss_sum/n:.4f} | G Loss: {g_loss_sum/n:.4f}")

    if (epoch + 1) % 5 == 0:
        generator.eval()
        with torch.no_grad():
            z = torch.randn(10, LATENT_DIM).to(device)
            samples = generator(z).cpu()
        fig, axes = plt.subplots(1, 10, figsize=(15, 2))
        for i, ax in enumerate(axes):
            ax.imshow(samples[i].squeeze(), cmap="gray")
            ax.axis("off")
        plt.suptitle(f"Generated Digits - Epoch {epoch+1}", fontsize=12)
        plt.tight_layout()
        plt.show()
        generator.train()

## 4. Generate New Digits

In [ ]:
generator.eval()
with torch.no_grad():
    z = torch.randn(20, LATENT_DIM).to(device)
    generated = generator(z).cpu()

fig, axes = plt.subplots(2, 10, figsize=(16, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(generated[i].squeeze(), cmap="gray")
    ax.axis("off")
plt.suptitle("GAN Generated Digits", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
z = torch.randn(1, LATENT_DIM).to(device)
with torch.no_grad():
    fake = generator(z).cpu().squeeze()

classifier.eval()
with torch.no_grad():
    pred = classifier(fake.unsqueeze(0).to(device))
    prob = torch.softmax(pred, dim=1)
    conf, digit = prob.max(1)

plt.figure(figsize=(4, 4))
plt.imshow(fake, cmap="gray")
plt.title(f"Generated digit: {digit.item()} (confidence: {conf.item()*100:.1f}%)", fontsize=12)
plt.axis("off")
plt.show()